In [97]:
import json
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import OneHotEncoder
import logging
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Lambda
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings
import tensorflow as tf
from tensorflow.keras import layers, models, metrics, optimizers
import glob
import pandas as pd
import matplotlib.pyplot as plt

In [98]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

# Detect device
def get_device():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                details = tf.config.experimental.get_device_details(gpu)
                device_name = details.get("device_name", "").upper()
                if "NVIDIA" in device_name:
                    logger.info("CUDA GPU detected. Using NVIDIA GPU.")
                    return 'GPU'
                elif "APPLE" in device_name or "METAL" in device_name:
                    logger.info("Metal GPU detected. Using Apple M-series GPU.")
                    return 'GPU'
            logger.info("Unknown GPU type detected. Defaulting to GPU.")
            return 'GPU'
        except Exception as e:
            logger.warning(f"Could not get GPU details: {e}")
            return 'GPU'
    else:
        logger.info("No compatible GPU detected. Using CPU.")
        tf.config.set_visible_devices([], 'GPU')
        return 'CPU'

device_type = get_device()


INFO:__main__:Metal GPU detected. Using Apple M-series GPU.


In [99]:


# Fixed dimensions
SEQ_LEN = 316
FEN_LEN = 70   # length of each FEN vector
MOVE_DIM = 5   # number of ints per move

def parse_tfrecord(example_proto):
    feature_desc = {
        'total_move_count': tf.io.FixedLenFeature([], tf.int64),
        'halfmove_counts': tf.io.VarLenFeature(tf.float32),
        'turn': tf.io.VarLenFeature(tf.int64),
        'error_codes': tf.io.VarLenFeature(tf.float32),
        'cp_scores': tf.io.VarLenFeature(tf.float32),
        'time_ratios': tf.io.VarLenFeature(tf.float32),
        'winning_chances': tf.io.VarLenFeature(tf.float32),
        'drawing_chances': tf.io.VarLenFeature(tf.float32),
        'losing_chances': tf.io.VarLenFeature(tf.float32),
        'check_flags': tf.io.VarLenFeature(tf.int64),
        'mate_flags': tf.io.VarLenFeature(tf.int64),
        'mate_in_n': tf.io.VarLenFeature(tf.float32),
        'encoded_moves': tf.io.VarLenFeature(tf.float32),
        'encoded_fens': tf.io.VarLenFeature(tf.float32),
        'estimated_time': tf.io.FixedLenFeature([], tf.float32),
        'black_mistakes': tf.io.FixedLenFeature([], tf.float32),
        'black_inaccuracies': tf.io.FixedLenFeature([], tf.float32),
        'black_blunders': tf.io.FixedLenFeature([], tf.float32),
        'white_mistakes': tf.io.FixedLenFeature([], tf.float32),
        'white_inaccuracies': tf.io.FixedLenFeature([], tf.float32),
        'white_blunders': tf.io.FixedLenFeature([], tf.float32),
        'outcome': tf.io.FixedLenFeature([], tf.int64),
        'white_elo': tf.io.FixedLenFeature([], tf.int64),
        'black_elo': tf.io.FixedLenFeature([], tf.int64),
    }

    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    def to_dense(key, dtype):
        t = parsed[key]
        if isinstance(t, tf.SparseTensor):
            t = tf.sparse.to_dense(t)
        return tf.cast(t, dtype)

    tf.cast(parsed['total_move_count'], tf.int64)

    # Convert sparse to dense
    hm = to_dense('halfmove_counts', tf.float32)
    trn = to_dense('turn', tf.float32)
    err = to_dense('error_codes', tf.float32)
    cp = to_dense('cp_scores', tf.float32)
    tr = to_dense('time_ratios', tf.float32)
    wc = to_dense('winning_chances', tf.float32)
    dc = to_dense('drawing_chances', tf.float32)
    lc = to_dense('losing_chances', tf.float32)
    cf = to_dense('check_flags', tf.float32)
    mf = to_dense('mate_flags', tf.float32)
    mi = to_dense('mate_in_n', tf.float32)
    mv_flat = to_dense('encoded_moves', tf.float32)
    fe_flat = to_dense('encoded_fens', tf.float32)
    
    # Reshape moves and FENs
    mv = tf.reshape(mv_flat, [-1, MOVE_DIM])
    fe = tf.reshape(fe_flat, [-1, FEN_LEN])
    
    # Ensure all tensors have the same length
    seq_len = tf.shape(mv)[0]
    

    
    # Make sure all features have same length as mv
    hm = tf.cond(tf.equal(tf.size(hm), seq_len), 
                 lambda: hm, 
                 lambda: tf.pad(hm, [[0, seq_len - tf.size(hm)]]))
    
    trn = tf.cond(tf.equal(tf.size(trn), seq_len), 
                 lambda: trn, 
                 lambda: tf.pad(trn, [[0, seq_len - tf.size(trn)]]))
    
    err = tf.cond(tf.equal(tf.size(err), seq_len), 
                 lambda: err, 
                 lambda: tf.pad(err, [[0, seq_len - tf.size(err)]]))
    
    cp = tf.cond(tf.equal(tf.size(cp), seq_len), 
                lambda: cp, 
                lambda: tf.pad(cp, [[0, seq_len - tf.size(cp)]]))
    
    tr = tf.cond(tf.equal(tf.size(tr), seq_len), 
                lambda: tr, 
                lambda: tf.pad(tr, [[0, seq_len - tf.size(tr)]]))
    
    wc = tf.cond(tf.equal(tf.size(wc), seq_len), 
                lambda: wc, 
                lambda: tf.pad(wc, [[0, seq_len - tf.size(wc)]]))
    
    dc = tf.cond(tf.equal(tf.size(dc), seq_len), 
                lambda: dc, 
                lambda: tf.pad(dc, [[0, seq_len - tf.size(dc)]]))
    
    lc = tf.cond(tf.equal(tf.size(lc), seq_len), 
                lambda: lc, 
                lambda: tf.pad(lc, [[0, seq_len - tf.size(lc)]]))
    
    cf = tf.cond(tf.equal(tf.size(cf), seq_len), 
                lambda: cf, 
                lambda: tf.pad(cf, [[0, seq_len - tf.size(cf)]]))
    
    mf = tf.cond(tf.equal(tf.size(mf), seq_len), 
                lambda: mf, 
                lambda: tf.pad(mf, [[0, seq_len - tf.size(mf)]]))
    
    mi = tf.cond(tf.equal(tf.size(mi), seq_len), 
                lambda: mi, 
                lambda: tf.pad(mi, [[0, seq_len - tf.size(mi)]]))
    
    # Reshape scalar features to match the concatenation dimension
    seq = tf.concat([
        tf.reshape(hm, [seq_len, 1]), 
        tf.reshape(trn, [seq_len, 1]), 
        tf.reshape(err, [seq_len, 1]), 
        tf.reshape(cp, [seq_len, 1]), 
        tf.reshape(tr, [seq_len, 1]),
        tf.reshape(wc, [seq_len, 1]), 
        tf.reshape(dc, [seq_len, 1]), 
        tf.reshape(lc, [seq_len, 1]), 
        tf.reshape(cf, [seq_len, 1]), 
        tf.reshape(mf, [seq_len, 1]), 
        tf.reshape(mi, [seq_len, 1]), 
        fe, 
        mv
    ], axis=-1)

    # Pad sequence to fixed length
    pad_len = SEQ_LEN - seq_len
    seq = tf.pad(seq, [[0, pad_len], [0, 0]])
    mask = tf.concat([
        tf.ones((seq_len,), tf.float32),
        tf.zeros((pad_len,), tf.float32)
    ], axis=0)

    meta = tf.stack([
        tf.cast(parsed['estimated_time'], tf.float32),
        tf.cast(parsed['black_mistakes'], tf.float32),
        tf.cast(parsed['black_inaccuracies'], tf.float32),
        tf.cast(parsed['black_blunders'], tf.float32),
        tf.cast(parsed['white_mistakes'], tf.float32),
        tf.cast(parsed['white_inaccuracies'], tf.float32),
        tf.cast(parsed['white_blunders'], tf.float32),
        tf.cast(parsed['outcome'], tf.float32),
    ])

    target = tf.stack([
        tf.cast(parsed['white_elo'], tf.float32) / 3000,
        tf.cast(parsed['black_elo'], tf.float32) / 3000,
    ])



    return (seq, meta, mask), target

In [100]:


# --- Custom Accuracy Metric ---
def elo_accuracy(threshold=100):
    def accuracy(y_true, y_pred):
        abs_diff = tf.abs(y_true - y_pred)
        within_threshold = tf.reduce_all(abs_diff <= threshold/3000, axis=-1)
        # Multiply by 100 here to get percentage
        return tf.reduce_mean(tf.cast(within_threshold, tf.float32)) * 100
    
    # Use the actual threshold value for the name
    # But convert to original ELO scale for clarity
    accuracy.__name__ = f'accuracy_within_{threshold}_elo'
    return accuracy

# --- Dataset Creation ---
def create_dataset(tfrecord_paths, batch_size=64, shuffle=True):
    ds = tf.data.TFRecordDataset(tfrecord_paths)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# --- Transformer Block ---
def transformer_block(x, mask, num_heads, ff_dim, dropout_rate=0.1):
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=x.shape[-1]
    )(x, x, attention_mask=mask[:, None, None, :])
    attn_output = layers.Dropout(dropout_rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    ff_output = layers.Dense(ff_dim, activation='relu')(out1)
    ff_output = layers.Dense(x.shape[-1])(ff_output)
    ff_output = layers.Dropout(dropout_rate)(ff_output)
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ff_output)
    return out2

# --- Model Architecture ---
def build_chess_model(input_shape=(SEQ_LEN, 86), d_model=256, num_heads=6, ff_dim=256, num_layers=6, meta_shape=(8,)):
    seq_inputs = layers.Input(shape=input_shape, name='seq_inputs')
    meta_inputs = layers.Input(shape=meta_shape, name='meta_inputs')
    mask_inputs = layers.Input(shape=(input_shape[0],), name='mask_inputs')

    x = layers.Dense(d_model)(seq_inputs)
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    pos_embedding = layers.Embedding(input_dim=input_shape[0], output_dim=d_model)(positions)
    x = x + pos_embedding

    for _ in range(num_layers):
        x = transformer_block(x, mask_inputs, num_heads, ff_dim)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Concatenate()([x, meta_inputs])
    x = layers.Dense(d_model // 2, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(2, activation='linear')(x)

    return models.Model(inputs=[seq_inputs, meta_inputs, mask_inputs], outputs=outputs)


In [101]:
# --- Combine TFRecords ---
def combine_tfrecords(input_folder, output_tfrecord):
    tfrecord_files = glob.glob(os.path.join(input_folder, "*.tfrecord"))
    
    if not tfrecord_files:
        raise ValueError(f"No TFRecord files found in {input_folder}")
    
    with tf.io.TFRecordWriter(output_tfrecord) as writer:
        for tfrecord_file in tfrecord_files:
            dataset = tf.data.TFRecordDataset(tfrecord_file)
            for record in dataset:
                writer.write(record.numpy())
    
    print(f"Combined {len(tfrecord_files)} TFRecord files into {output_tfrecord}")
    return output_tfrecord

# --- Save and Plot Losses ---
def save_and_plot_losses(history, csv_file='losses.csv', plot_file='loss_plot.png'):
    # Extract losses
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(train_loss) + 1)
    
    # Print losses
    print("\nLosses per Epoch:")
    print("Epoch | Train Loss | Validation Loss")
    print("------|------------|----------------")
    for epoch, t_loss, v_loss in zip(epochs, train_loss, val_loss):
        print(f"{epoch:5d} | {t_loss:10.4f} | {v_loss:15.4f}")
    
    # Save to CSV
    df = pd.DataFrame({
        'Epoch': epochs,
        'Train_Loss': train_loss,
        'Validation_Loss': val_loss
    })
    df.to_csv(csv_file, index=False)
    print(f"\nSaved losses to {csv_file}")
    
    # Plot losses
    plt.figure(figsize=(8, 6))
    plt.plot(epochs, train_loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss vs. Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.savefig(plot_file)
    print(f"Saved loss plot to {plot_file}")


In [102]:

# --- Train with Split ---
def train_with_split(tfrecord_file,patience=5, batch_size=64, epochs=50, train_split=0.8):
    dataset = tf.data.TFRecordDataset(tfrecord_file)
    num_examples = sum(1 for _ in dataset)
    print(f"Total dataset size: {num_examples} examples")
    
    train_size = int(train_split * num_examples)
    val_size = num_examples - train_size
    print(f"Training size: {train_size} examples, Validation size: {val_size} examples")
    
    dataset = dataset.shuffle(buffer_size=1000)
    train_ds = dataset.take(train_size)
    val_ds = dataset.skip(train_size)
    
    train_ds = train_ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    val_ds = val_ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    
    def train_and_evaluate_modified(train_ds, val_ds, batch_size, epochs):
        for (seq, meta, mask), target in train_ds.take(1):
            input_shape = seq.shape[1:]
            meta_shape = meta.shape[1:]
            print("Model input shapes:", input_shape, meta_shape, mask.shape[1:])
        
        model = build_chess_model(input_shape=input_shape, meta_shape=meta_shape)
        model.compile(
            optimizer=optimizers.Adam(learning_rate=0.0001),
            loss='mse',
            metrics=['mae', elo_accuracy(threshold=100)]
        )
        
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=patience,
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-6
            )
        ]
        
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )
        
        eval_metrics = model.evaluate(val_ds, return_dict=True)
        print("Final Validation Metrics:", eval_metrics)
        
        model.save('chess_elo_model.keras')
        
        # Save and plot losses
        save_and_plot_losses(history)
        
        return model, history
    
    model, history = train_and_evaluate_modified(train_ds, val_ds, batch_size, epochs)
    
    
    return model, history



In [103]:
def parse_tfrecord_no_target(example_proto):
    feature_desc = {
        'total_move_count': tf.io.FixedLenFeature([], tf.int64),
        'halfmove_counts': tf.io.VarLenFeature(tf.float32),
        'turn': tf.io.VarLenFeature(tf.int64),
        'error_codes': tf.io.VarLenFeature(tf.float32),
        'cp_scores': tf.io.VarLenFeature(tf.float32),
        'time_ratios': tf.io.VarLenFeature(tf.float32),
        'winning_chances': tf.io.VarLenFeature(tf.float32),
        'drawing_chances': tf.io.VarLenFeature(tf.float32),
        'losing_chances': tf.io.VarLenFeature(tf.float32),
        'check_flags': tf.io.VarLenFeature(tf.int64),
        'mate_flags': tf.io.VarLenFeature(tf.int64),
        'mate_in_n': tf.io.VarLenFeature(tf.float32),
        'encoded_moves': tf.io.VarLenFeature(tf.float32),
        'encoded_fens': tf.io.VarLenFeature(tf.float32),
        'estimated_time': tf.io.FixedLenFeature([], tf.float32),
        'black_mistakes': tf.io.FixedLenFeature([], tf.float32),
        'black_inaccuracies': tf.io.FixedLenFeature([], tf.float32),
        'black_blunders': tf.io.FixedLenFeature([], tf.float32),
        'white_mistakes': tf.io.FixedLenFeature([], tf.float32),
        'white_inaccuracies': tf.io.FixedLenFeature([], tf.float32),
        'white_blunders': tf.io.FixedLenFeature([], tf.float32),
        'outcome': tf.io.FixedLenFeature([], tf.int64),
        # no white_elo or black_elo
    }

    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    def to_dense(key, dtype):
        t = parsed[key]
        if isinstance(t, tf.SparseTensor):
            t = tf.sparse.to_dense(t)
        return tf.cast(t, dtype)

    # Same logic as your original parse_tfrecord, but no "target"
    hm = to_dense('halfmove_counts', tf.float32)
    trn = to_dense('turn', tf.float32)
    err = to_dense('error_codes', tf.float32)
    cp = to_dense('cp_scores', tf.float32)
    tr = to_dense('time_ratios', tf.float32)
    wc = to_dense('winning_chances', tf.float32)
    dc = to_dense('drawing_chances', tf.float32)
    lc = to_dense('losing_chances', tf.float32)
    cf = to_dense('check_flags', tf.float32)
    mf = to_dense('mate_flags', tf.float32)
    mi = to_dense('mate_in_n', tf.float32)
    mv_flat = to_dense('encoded_moves', tf.float32)
    fe_flat = to_dense('encoded_fens', tf.float32)

    mv = tf.reshape(mv_flat, [-1, MOVE_DIM])
    fe = tf.reshape(fe_flat, [-1, FEN_LEN])

    seq_len = tf.shape(mv)[0]

    def pad_to_len(tensor):
        return tf.cond(
            tf.equal(tf.size(tensor), seq_len),
            lambda: tensor,
            lambda: tf.pad(tensor, [[0, seq_len - tf.size(tensor)]]))

    features = [hm, trn, err, cp, tr, wc, dc, lc, cf, mf, mi]
    features = [pad_to_len(f) for f in features]
    reshaped = [tf.reshape(f, [seq_len, 1]) for f in features]
    seq = tf.concat(reshaped + [fe, mv], axis=-1)

    pad_len = SEQ_LEN - seq_len
    seq = tf.pad(seq, [[0, pad_len], [0, 0]])
    mask = tf.concat([tf.ones((seq_len,), tf.float32), tf.zeros((pad_len,), tf.float32)], axis=0)

    meta = tf.stack([
        tf.cast(parsed['estimated_time'], tf.float32),
        tf.cast(parsed['black_mistakes'], tf.float32),
        tf.cast(parsed['black_inaccuracies'], tf.float32),
        tf.cast(parsed['black_blunders'], tf.float32),
        tf.cast(parsed['white_mistakes'], tf.float32),
        tf.cast(parsed['white_inaccuracies'], tf.float32),
        tf.cast(parsed['white_blunders'], tf.float32),
        tf.cast(parsed['outcome'], tf.float32),
    ])

    return (seq, meta, mask)

In [104]:

def split_tfrecord_into_train_test(input_path, train_output_path, test_output_path, train_ratio=0.8):
    raw_dataset = tf.data.TFRecordDataset(input_path)
    total_records = sum(1 for _ in raw_dataset)

    train_count = int(train_ratio * total_records)
    print(f"Total records: {total_records}, Train: {train_count}, Test: {total_records - train_count}")

    # Reset the dataset iterator
    raw_dataset = tf.data.TFRecordDataset(input_path)

    # Split
    train_dataset = raw_dataset.take(train_count)
    test_dataset = raw_dataset.skip(train_count)

    # Write train split
    with tf.io.TFRecordWriter(train_output_path) as train_writer:
        for record in train_dataset:
            train_writer.write(record.numpy())

    # Write test split
    with tf.io.TFRecordWriter(test_output_path) as test_writer:
        for record in test_dataset:
            test_writer.write(record.numpy())

    print(f"TFRecord split complete.\nTrain: {train_output_path}\nTest: {test_output_path}")

In [105]:
def evaluate_on_test_record(model, test_tfrecord_path, batch_size=64):
    test_ds = tf.data.TFRecordDataset(test_tfrecord_path)\
        .map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)\
        .batch(batch_size)\
        .prefetch(tf.data.AUTOTUNE)

    print("\n--- Test Set Evaluation ---")
    test_metrics = model.evaluate(test_ds, return_dict=True)
    print("Test Metrics:", test_metrics)
    return test_metrics

In [106]:
def predict_on_test_record(model, test_tfrecord_path, batch_size=64):
    test_ds = tf.data.TFRecordDataset(test_tfrecord_path)\
        .map(parse_tfrecord_no_target, num_parallel_calls=tf.data.AUTOTUNE)\
        .batch(batch_size)\
        .prefetch(tf.data.AUTOTUNE)

    print("\n--- Running Inference on Test Set (no targets) ---")
    predictions = model.predict(test_ds)
    print("Predictions shape:", predictions.shape)
    return predictions

In [107]:
def handle_test_record(model, test_tfrecord_path, batch_size=64):
    raw_dataset = tf.data.TFRecordDataset(test_tfrecord_path)

    try:
        # Peek at the first record
        example = next(iter(raw_dataset.take(1)))
        example_dict = tf.io.parse_single_example(example, {
            'white_elo': tf.io.FixedLenFeature([], tf.int64),
            'black_elo': tf.io.FixedLenFeature([], tf.int64),
        })

        # If the above succeeds, targets exist — run full evaluation
        print("✅ Target found in test set. Running evaluation.")
        return evaluate_on_test_record(model, test_tfrecord_path, batch_size)

    except tf.errors.InvalidArgumentError:
        # Target fields are missing — use prediction mode
        print("⚠️ No target found in test set. Running prediction only.")
        return predict_on_test_record(model, test_tfrecord_path, batch_size)

In [108]:
if __name__ == "__main__":
    # Split the combined TFRecord
    input_folder = 'tfrecord'
    combined_tfrecord_path = f'{input_folder}/combined.tfrecord'
    train_output_path = f'{input_folder}/train.tfrecord'
    test_output_path = f'{input_folder}/test.tfrecord'
    import os
    if os.path.exists(combined_tfrecord_path): 
        os.remove(combined_tfrecord_path)
    if os.path.exists(train_output_path): 
        os.remove(train_output_path)
    if os.path.exists(test_output_path): 
        os.remove(test_output_path)
    
    combined_tfrecord = combine_tfrecords(input_folder, combined_tfrecord_path)

    split_tfrecord_into_train_test(
        combined_tfrecord_path,
        train_output_path,
        test_output_path,
        train_ratio=0.8
    )

    # Train on train_only.tfrecord (your existing function handles train/val split)
    model, history = train_with_split(train_output_path, patience=3, batch_size=64, epochs=30, train_split=0.9)
    print('training completed')
    handle_test_record(model, test_output_path)

Combined 13 TFRecord files into tfrecord/combined.tfrecord
Total records: 55919, Train: 44735, Test: 11184
TFRecord split complete.
Train: tfrecord/train.tfrecord
Test: tfrecord/test.tfrecord
Total dataset size: 44735 examples
Training size: 40261 examples, Validation size: 4474 examples
Model input shapes: (316, 86) (8,) (316,)
Epoch 1/30
630/630 ━━━━━━━━━━━━━━━━━━━━ 3131s 5s/step - accuracy_within_100_elo: 0.3297 - loss: 0.4058 - mae: 0.4693 - val_accuracy_within_100_elo: 5.7552 - val_loss: 0.0202 - val_mae: 0.1146 - learning_rate: 1.0000e-04
Epoch 2/30
630/630 ━━━━━━━━━━━━━━━━━━━━ 2951s 5s/step - accuracy_within_100_elo: 0.3782 - loss: 0.2245 - mae: 0.3752 - val_accuracy_within_100_elo: 0.1761 - val_loss: 0.0227 - val_mae: 0.1221 - learning_rate: 1.0000e-04
Epoch 3/30
630/630 ━━━━━━━━━━━━━━━━━━━━ 2914s 5s/step - accuracy_within_100_elo: 0.3548 - loss: 0.2220 - mae: 0.3741 - val_accuracy_within_100_elo: 11.8049 - val_loss: 0.0120 - val_mae: 0.0887 - learning_rate: 1.0000e-04
Epoch 4/

In [ ]:
# # --- Main Execution ---
# if __name__ == "__main__":
#     model,history = train_with_split(combined_tfrecord_path,patience=3, batch_size=64, epochs=30, train_split=0.9)